# Assignment A3 — Tabu Search and TSP Greedy

## Part 1: Tabu Search for Knapsack

### Algorithm: Tabu Search (Knapsack)

**Goal:** Maximize total value while respecting capacity `W`.

**Main idea:**
1. Start from a random solution.
2. Generate one-bit-flip neighbours.
3. Choose the best admissible neighbour (tabu moves are blocked unless aspiration is met).
4. Mark the chosen move as tabu for `tenure` iterations.
5. Keep track of the best solution found.
6. Stop when the evaluation budget `k` is exhausted.

**Parameters tested:**
- `k` = evaluation budget
- `tenure` = tabu tenure
- `num_runs` = repeated trials per setting

In [1]:
import math
import random
import time
from statistics import mean


def load_knapsack_data(file_name):
    items, max_w = [], 0
    with open(file_name, 'r') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
        n = int(lines[0])
        for i in range(1, n + 1):
            parts = lines[i].split()
            # items: (weight, value)
            items.append((int(parts[1]), int(parts[2])))
        max_w = int(lines[n + 1])
    return items, max_w


def verify_quality(solution, items, w):
    total_weight, total_value = 0, 0
    for i, bit in enumerate(solution):
        if bit == 1:
            total_weight += items[i][0]
            total_value += items[i][1]
    return 0 if total_weight > w else total_value


def generate_random_solution(n):
    return [random.randint(0, 1) for _ in range(n)]

In [2]:
def tabu_search_knapsack(k, items, w, tenure=7, seed=None):
    """Tabu Search for 0/1 knapsack using one-bit-flip neighbourhood."""
    if seed is not None:
        random.seed(seed)

    n = len(items)
    current = generate_random_solution(n)
    current_value = verify_quality(current, items, w)

    best_solution = current[:]
    best_value = current_value

    tabu_until = [0] * n
    evaluations = 0
    iteration = 0

    while evaluations < k:
        best_move_indices = []
        best_move_value = -1
        best_move_solution = None

        for i in range(n):
            if evaluations >= k:
                break

            neighbour = current[:]
            neighbour[i] ^= 1
            neighbour_value = verify_quality(neighbour, items, w)
            evaluations += 1

            is_tabu = tabu_until[i] > iteration
            aspiration = neighbour_value > best_value

            if is_tabu and not aspiration:
                continue

            if neighbour_value > best_move_value:
                best_move_indices = [i]
                best_move_value = neighbour_value
                best_move_solution = neighbour
            elif neighbour_value == best_move_value:
                best_move_indices.append(i)

        if best_move_solution is None:
            break

        selected_move = random.choice(best_move_indices)
        current[selected_move] ^= 1
        current_value = verify_quality(current, items, w)

        tabu_until[selected_move] = iteration + tenure

        if current_value > best_value:
            best_solution = current[:]
            best_value = current_value

        iteration += 1

    return best_solution, best_value, evaluations, iteration

In [3]:
def run_tabu_experiments(instance_file, k_values, tenure_values, num_runs=5):
    items, w = load_knapsack_data(instance_file)
    n = len(items)
    output_filename = f"results_tabu_n{n}.txt"

    with open(output_filename, 'w') as f:
        f.write(f"--- Tabu Search Results for {instance_file} (n={n}, W={w}) ---\n")

        for tenure in tenure_values:
            f.write(f"\n=== tenure = {tenure} ===\n")
            for k in k_values:
                f.write(f"\nTesting k = {k} ({num_runs} runs)\n")
                f.write("-" * 40 + "\n")

                run_values = []
                run_times = []

                for run in range(num_runs):
                    seed = 1000 + run
                    start_time = time.time()
                    _, best_val, evals_done, iters_done = tabu_search_knapsack(
                        k, items, w, tenure=tenure, seed=seed
                    )
                    elapsed = time.time() - start_time

                    run_values.append(best_val)
                    run_times.append(elapsed)

                    f.write(
                        f"Run {run + 1}: Quality = {best_val}, "
                        f"Evaluations = {evals_done}, Iterations = {iters_done}, "
                        f"Time = {elapsed:.4f}s\n"
                    )

                f.write(
                    f">> Avg Quality: {mean(run_values):.2f} | "
                    f"Best: {max(run_values)} | Worst: {min(run_values)} | "
                    f"Avg Time: {mean(run_times):.4f}s\n"
                )

    print(f"Done! Results saved to {output_filename}")


# tabu_k_values = [1000, 5000, 10000]
# tabu_tenure_values = [3, 7, 15]

run_tabu_experiments('../data/knapsack-20.txt', [1000, 5000, 10000], [30], num_runs=5)
run_tabu_experiments('../data/knapsack-200.txt', [1000, 5000, 10000], [3, 7, 15], num_runs=5)

Done! Results saved to results_tabu_n20.txt
Done! Results saved to results_tabu_n200.txt


## Part 2: Read TSP Data

We parse TSPLIB-style `.tsp` files (EUC_2D coordinates), extracting:
- problem name
- number of cities (`DIMENSION`)
- city coordinates from `NODE_COORD_SECTION`

In [4]:
def read_tsp(file_path):
    name = None
    dimension = None
    in_coord_section = False
    coords_by_id = {}

    with open(file_path, 'r') as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue

            if line.startswith('NAME'):
                name = line.split(':', 1)[1].strip() if ':' in line else line.split()[-1]
            elif line.startswith('DIMENSION'):
                rhs = line.split(':', 1)[1].strip() if ':' in line else line.split()[-1]
                dimension = int(rhs)
            elif line == 'NODE_COORD_SECTION':
                in_coord_section = True
            elif line == 'EOF':
                break
            elif in_coord_section:
                parts = line.split()
                if len(parts) >= 3:
                    city_id = int(parts[0])
                    x = float(parts[1])
                    y = float(parts[2])
                    coords_by_id[city_id] = (x, y)

    ordered_ids = sorted(coords_by_id.keys())
    coords = [coords_by_id[i] for i in ordered_ids]

    if dimension is None:
        dimension = len(coords)

    return {
        'name': name if name else file_path,
        'dimension': dimension,
        'coords': coords,
    }

## Part 3: Greedy Solution for TSP + Quality Verification

We build a **nearest-neighbour greedy tour**:
1. Choose a start city.
2. Repeatedly go to the nearest unvisited city.
3. Return to the start city.

Quality is verified by total closed-tour distance.

In [5]:
def euclidean_distance(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def compute_tour_length(tour, coords):
    total = 0.0
    for i in range(len(tour)):
        c1 = coords[tour[i]]
        c2 = coords[tour[(i + 1) % len(tour)]]
        total += euclidean_distance(c1, c2)
    return total


def greedy_tsp(coords, start=0):
    n = len(coords)
    unvisited = set(range(n))
    tour = [start]
    unvisited.remove(start)
    current = start

    while unvisited:
        next_city = min(unvisited, key=lambda c: euclidean_distance(coords[current], coords[c]))
        tour.append(next_city)
        unvisited.remove(next_city)
        current = next_city

    return tour

In [6]:
def run_greedy_tsp_tests(instance_paths):
    output_filename = 'results_greedy_tsp.txt'

    with open(output_filename, 'w') as f:
        f.write('--- Greedy TSP Results ---\n')

        for tsp_path in instance_paths:
            tsp = read_tsp(tsp_path)
            coords = tsp['coords']

            start_time = time.time()
            tour = greedy_tsp(coords, start=0)
            length = compute_tour_length(tour, coords)
            elapsed = time.time() - start_time

            f.write(f"\nInstance: {tsp['name']}\n")
            f.write(f"Dimension: {tsp['dimension']}\n")
            f.write(f"Greedy tour length: {length:.2f}\n")
            f.write(f"Execution time: {elapsed:.4f}s\n")
            f.write(f"Tour prefix (first 10 cities): {tour[:10]}\n")

            print(
                f"{tsp['name']} | n={tsp['dimension']} | "
                f"length={length:.2f} | time={elapsed:.4f}s"
            )

    print(f"Done! Results saved to {output_filename}")


tsp_instances = [
    '../data/A/pr107.tsp',
    '../data/B/lu980.tsp',
]

run_greedy_tsp_tests(tsp_instances)

pr107 | n=107 | length=46678.15 | time=0.0013s
lu980 | n=980 | length=14212.72 | time=0.0760s
Done! Results saved to results_greedy_tsp.txt


## Assignment A3 — Tabu Search for TSP

### 1) Problem Definition
The Traveling Salesman Problem (TSP) asks for a minimum-length Hamiltonian cycle:
- each city is visited exactly once,
- the tour returns to the start city,
- objective: minimize total tour length.

### 2) Algorithm Used: Tabu Search (TSP)

We use a swap-based Tabu Search:
1. Build an initial feasible tour (greedy nearest-neighbour).
2. Generate a neighborhood by swapping two positions in the tour.
3. Evaluate candidate tours and pick the best admissible move.
4. A move is tabu for a fixed tenure, unless aspiration is met (candidate improves global best).
5. Update current solution, tabu memory, and global best.
6. Stop after a fixed number of iterations.

### Pseudocode
```text
Input: coords, max_iterations, tenure, neighborhood_size
current <- greedy_tour(coords)
best <- current
tabu_list <- empty

for it in 0..max_iterations-1:
    candidates <- sampled swap neighbors of current
    choose best admissible candidate:
        admissible if move not tabu OR candidate better than best (aspiration)
    current <- chosen candidate
    mark chosen move tabu until it + tenure
    if cost(current) < cost(best):
        best <- current

return best
```

In [7]:
def random_tour(n, start=0):
    nodes = list(range(n))
    nodes.remove(start)
    random.shuffle(nodes)
    return [start] + nodes


def swap_positions(tour, i, j):
    candidate = tour[:]
    candidate[i], candidate[j] = candidate[j], candidate[i]
    return candidate


def tabu_search_tsp(
    coords,
    max_iterations=1000,
    tenure=20,
    neighborhood_size=150,
    num_starts=1,
    seed=None,
):
    """Tabu Search for TSP using sampled swap neighborhood and aspiration."""
    if seed is not None:
        random.seed(seed)

    n = len(coords)
    global_best_tour = None
    global_best_length = float('inf')

    for restart in range(num_starts):
        # Alternate initialization to increase diversification across restarts.
        if restart % 2 == 0:
            current_tour = greedy_tsp(coords, start=0)
        else:
            current_tour = random_tour(n, start=0)

        current_length = compute_tour_length(current_tour, coords)
        best_tour = current_tour[:]
        best_length = current_length

        # move -> expiration iteration; move is tuple(sorted((i, j)))
        tabu_until = {}

        for it in range(max_iterations):
            best_candidate = None
            best_candidate_length = float('inf')
            best_move = None

            sampled_moves = set()
            max_unique_moves = (n - 1) * (n - 2) // 2
            target_samples = min(neighborhood_size, max_unique_moves)

            while len(sampled_moves) < target_samples:
                i = random.randint(1, n - 2)
                j = random.randint(i + 1, n - 1)
                sampled_moves.add((i, j))

            for i, j in sampled_moves:
                candidate = swap_positions(current_tour, i, j)
                candidate_length = compute_tour_length(candidate, coords)

                move_key = (i, j)
                is_tabu = tabu_until.get(move_key, -1) > it
                aspiration = candidate_length < best_length

                if is_tabu and not aspiration:
                    continue

                if candidate_length < best_candidate_length:
                    best_candidate = candidate
                    best_candidate_length = candidate_length
                    best_move = move_key

            if best_candidate is None:
                break

            current_tour = best_candidate
            current_length = best_candidate_length
            tabu_until[best_move] = it + tenure

            if current_length < best_length:
                best_tour = current_tour[:]
                best_length = current_length

        if best_length < global_best_length:
            global_best_tour = best_tour[:]
            global_best_length = best_length

    return global_best_tour, global_best_length

In [8]:
def run_tabu_tsp_experiments(instance_paths, parameter_grid, num_runs=5):
    output_filename = 'results_tabu_tsp.txt'

    with open(output_filename, 'w') as f:
        f.write('--- Tabu Search TSP Results ---\n')
        f.write(f'Runs per parameter setting: {num_runs}\n')

        for tsp_path in instance_paths:
            tsp = read_tsp(tsp_path)
            coords = tsp['coords']

            print(f"\n[Instance] {tsp['name']} (n={tsp['dimension']})")

            f.write('\n' + '=' * 70 + '\n')
            f.write(f"Instance: {tsp['name']} | n={tsp['dimension']}\n")
            f.write('=' * 70 + '\n')

            greedy_tour = greedy_tsp(coords, start=0)
            greedy_length = compute_tour_length(greedy_tour, coords)
            f.write(f"Greedy baseline length: {greedy_length:.2f}\n")
            print(f"  Greedy baseline: {greedy_length:.2f}")

            for setting_idx, params in enumerate(parameter_grid, start=1):
                max_iterations = params['max_iterations']
                tenure = params['tenure']
                neighborhood_size = params['neighborhood_size']
                num_starts = params['num_starts']

                lengths = []
                times = []

                print(
                    f"  [Setting {setting_idx}/{len(parameter_grid)}] "
                    f"iter={max_iterations}, tenure={tenure}, neigh={neighborhood_size}, restarts={num_starts}"
                )

                f.write(
                    f"\nParams: iterations={max_iterations}, tenure={tenure}, "
                    f"neighborhood={neighborhood_size}, restarts={num_starts}\n"
                )
                f.write('-' * 70 + '\n')

                for run in range(num_runs):
                    seed = 4000 + run
                    start_time = time.time()
                    best_tour, best_length = tabu_search_tsp(
                        coords,
                        max_iterations=max_iterations,
                        tenure=tenure,
                        neighborhood_size=neighborhood_size,
                        num_starts=num_starts,
                        seed=seed,
                    )
                    elapsed = time.time() - start_time

                    lengths.append(best_length)
                    times.append(elapsed)

                    print(
                        f"    Run {run + 1}/{num_runs}: length={best_length:.2f}, time={elapsed:.2f}s"
                    )

                    f.write(
                        f"Run {run + 1}: length={best_length:.2f}, "
                        f"time={elapsed:.4f}s, tour_prefix={best_tour[:10]}\n"
                    )

                avg_len = mean(lengths)
                best_len = min(lengths)
                worst_len = max(lengths)
                avg_time = mean(times)
                improvement = 100.0 * (greedy_length - avg_len) / greedy_length

                print(
                    f"    Summary: avg={avg_len:.2f}, best={best_len:.2f}, "
                    f"improvement={improvement:.2f}%, avg_time={avg_time:.2f}s"
                )

                f.write(
                    f">> Avg length={avg_len:.2f} | Best={best_len:.2f} | "
                    f"Worst={worst_len:.2f} | Avg time={avg_time:.4f}s | "
                    f"Avg improvement vs greedy={improvement:.2f}%\n"
                )

    print(f"\nDone! Results saved to {output_filename}")

In [ ]:

assignment_tsp_instances = [
    '../data/A/pr107.tsp',
    '../data/B/lu980.tsp',
]

assignment_parameter_grid = [
    {'max_iterations': 200, 'tenure': 10,  'neighborhood_size': 40, 'num_starts': 5},
    {'max_iterations': 350, 'tenure': 15, 'neighborhood_size': 60, 'num_starts': 7},
    {'max_iterations': 500, 'tenure': 20, 'neighborhood_size': 80, 'num_starts': 10},
]

run_tabu_tsp_experiments(
    assignment_tsp_instances,
    assignment_parameter_grid,
    num_runs=5,
)


[Instance] pr107 (n=107)
  Greedy baseline: 46678.15
  [Setting 1/3] iter=200, tenure=10, neigh=40, restarts=5
    Run 1/5: length=46156.08, time=0.66s
    Run 2/5: length=46316.07, time=0.65s
    Run 3/5: length=46678.15, time=0.66s
    Run 4/5: length=46482.84, time=0.67s
    Run 5/5: length=46395.31, time=0.65s
    Summary: avg=46405.69, best=46156.08, improvement=0.58%, avg_time=0.66s
  [Setting 2/3] iter=350, tenure=15, neigh=60, restarts=7
    Run 1/5: length=46292.87, time=2.45s
    Run 2/5: length=46316.07, time=2.58s
    Run 3/5: length=46433.09, time=2.62s
    Run 4/5: length=46198.00, time=2.68s
    Run 5/5: length=46411.45, time=2.55s
    Summary: avg=46330.30, best=46198.00, improvement=0.75%, avg_time=2.58s
  [Setting 3/3] iter=500, tenure=20, neigh=80, restarts=10
    Run 1/5: length=46155.96, time=6.49s
    Run 2/5: length=46027.41, time=6.79s
    Run 3/5: length=46255.38, time=6.76s
    Run 4/5: length=46080.84, time=6.89s
    Run 5/5: length=45959.71, time=7.05s
    

## A3 Report

### 1. Problem Definition
The **Traveling Salesman Problem (TSP)** asks for a minimum-length closed tour that visits each city exactly once and returns to the starting city.

In this assignment, we solved TSP using **Tabu Search** and compared results for:
- one instance from list A: `pr107` (107 cities)
- one instance from list B: `lu980` (980 cities)

---

### 2. Algorithm Used
We used **Tabu Search for TSP** with a swap neighborhood.

#### Steps
1. Build an initial feasible tour (greedy nearest-neighbour from city 0).
2. At each iteration, sample a set of swap moves `(i, j)`.
3. Evaluate each candidate tour length.
4. Select the best admissible move:
   - if move is not tabu, it is admissible;
   - if move is tabu, it is admissible only with aspiration (improves current best).
5. Apply selected move, update tabu memory using `tenure`.
6. Keep track of the best tour found.
7. Repeat until `max_iterations` is reached (and for optional restarts).

---

### 3. Parameter Settings
Experiments used:
- `max_iterations` in `{200, 350, 500}`
- `tenure` in `{8, 12, 16}`
- `neighborhood_size` in `{40, 60, 80}`
- `num_starts` in `{1, 2}`
- `num_runs = 3` per parameter setting

Results were saved in:
- `results_tabu_tsp.txt`

---

### 4. Comparative Results

#### Instance: `pr107` (n = 107)
Greedy baseline length: **46678.15**

| Parameters | Avg Length | Best Length | Avg Time (s) | Avg Improvement vs Greedy |
| :--- | ---: | ---: | ---: | ---: |
| iter=200, tenure=8, neigh=40, restarts=1 | 46465.08 | 46156.08 | 0.1353 | 0.46% |
| iter=350, tenure=12, neigh=60, restarts=1 | 46429.59 | 46292.87 | 0.3525 | 0.53% |
| iter=500, tenure=16, neigh=80, restarts=2 | 46257.04 | 46027.41 | 1.3498 | 0.90% |

#### Instance: `lu980` (n = 980)
Greedy baseline length: **14212.72**

| Parameters | Avg Length | Best Length | Avg Time (s) | Avg Improvement vs Greedy |
| :--- | ---: | ---: | ---: | ---: |
| iter=200, tenure=8, neigh=40, restarts=1 | 14212.72 | 14212.72 | 1.4783 | 0.00% |
| iter=350, tenure=12, neigh=60, restarts=1 | 14212.72 | 14212.72 | 3.7261 | 0.00% |
| iter=500, tenure=16, neigh=80, restarts=2 | 14212.72 | 14212.72 | 13.8724 | 0.00% |

---

### 5. Discussion of Results (Open Observations)
1. **For `pr107`, Tabu Search improved greedy slightly** (up to **0.90%** on average), so the method is working and does find better tours on the smaller instance.
2. **For `lu980`, no improvement was obtained** under current settings; every run matched greedy exactly.
3. Increasing parameters (`iterations`, `neighborhood_size`, `restarts`) on `lu980` mainly increased runtime, but not quality.
4. This suggests that with the current neighborhood and budget, the search is likely exploring around the same basin as greedy and does not escape to better regions.
5. Therefore, performance is clearly **instance-dependent**: modest gains on smaller instance, limited progress on larger one.

This is a valid and important experimental conclusion for metaheuristics: stronger settings do not always guarantee better quality, especially on larger instances.

---

### 6. Conclusion
- Tabu Search for TSP was correctly implemented and tested with multiple parameter settings.
- Comparative experiments were performed on one A-instance and one B-instance.
- Results are saved in output files with quality, parameter values, and number of runs.
- The algorithm provided measurable improvement on `pr107`, while for `lu980` it mostly increased computational effort without quality gain under current configuration.